<a href="https://colab.research.google.com/github/DSPagan/llms-time-complexity/blob/main/notebooks/llm_complexity_estimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prepare the datasets

In [ ]:
user = "DSPagan"
repo = "llms-time-complexity"
src_dir = "data"
train_file = "train_data.jsonl"
test_file = "test_data.jsonl"

url_1 = f"https://raw.githubusercontent.com/{user}/{repo}/main/{src_dir}/{train_file}"
url_2 = f"https://raw.githubusercontent.com/{user}/{repo}/main/{src_dir}/{test_file}"

!wget --no-cache --backups=1 {url_1}
!wget --no-cache --backups=1 {url_2}

# Install dependencies

In [ ]:
# Unsloth pulls its own compatible stack; Colab already provides a CUDA-enabled PyTorch.
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo

# Import libraries

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import json

# Load model

In [ ]:
model_path = "unsloth/Meta-Llama-3.1-8B-Instruct"
max_seq_length = 2048

# Load the model and tokenizer (4-bit quantized)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    dtype = None,
)

# Apply the correct chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

# Fine-tuning with QLoRA

In [ ]:
train_data_path = "train_data.jsonl"
output_dir = "outputs"
num_epochs = 2
lora_r = 16
max_seq_length = 2048

# Load the training data
with open(train_data_path, "r") as f:
    train_data = [json.loads(line.strip()) for line in f]

def build_prompt(src):
    return (
        "Analyze the time complexity of the following code.\n"
        "Choose exactly one of the following options: O(1), O(logn), O(n), "
        "O(nlogn), O(n^2), O(n^3) or exponential (O(2^n), O(3^n), etc.).\n"
        "Give the time complexity of the code:\n"
        f"{src}"
    )

# Build a chat dataset: one user turn (the prompt) + one assistant turn (the label)
rows = [
    {"conversations": [
        {"role": "user", "content": build_prompt(item["src"])},
        {"role": "assistant", "content": item["complexity"]},
    ]}
    for item in train_data
]

def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in examples["conversations"]
    ]
    return {"text": texts}

dataset = Dataset.from_list(rows).map(formatting_prompts_func, batched=True)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_r,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = num_epochs,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = output_dir,
        report_to = "none",
    ),
)

# Train only on the assistant's response tokens
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

trainer_stats = trainer.train()

# Inference

In [ ]:
import os

test_data_path = "test_data.jsonl"
save_path = "outputs/test_results.jsonl"
max_new_tokens = 1024

FastLanguageModel.for_inference(model)

def build_prompt(src):
    return (
        "Analyze the time complexity of the following code.\n"
        "Choose exactly one of the following options: O(1), O(logn), O(n), "
        "O(nlogn), O(n^2), O(n^3) or exponential (O(2^n), O(3^n), etc.).\n"
        "Give the time complexity of the code:\n"
        f"{src}"
    )

with open(test_data_path, "r") as f:
    test_data = [json.loads(line.strip()) for line in f]

os.makedirs(os.path.dirname(save_path), exist_ok=True)

with open(save_path, "w") as out_file:
    for item in test_data:
        messages = [{"role": "user", "content": build_prompt(item["src"])}]

        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        ).to("cuda")

        if len(inputs[0]) > max_seq_length:
            continue

        tokens = model.generate(
            input_ids=inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            no_repeat_ngram_size=4,
        )

        result = tokenizer.decode(tokens[0], skip_special_tokens=True)
        result = result.split("assistant")[-1].strip()

        entry = {"src": item["src"], "complexity": item["complexity"], "model": result}
        out_file.write(json.dumps(entry) + "\n")